# 01 — Classificatore a feature (TF-IDF + regressione logistica)

Sul dataset finale (`1-data-cleaning/03_dataset_finale`): solo il **testo degli articoli**.

Fold **per gruppo** (la valutazione principale: tutte le testate di un gruppo, cioè rete o stesso owner, nello
stesso fold) e **per testata** (confronto: i gruppi possono essere spezzati). Un modello per fold: vocabolario e
IDF si stimano **solo sul training** del fold, il test passa solo da `transform`. Iperparametri fissi.

Con `TAGLIO_TOKEN` il testo si taglia ai primi N token del tokenizer di ModernBERT: lo stesso testo che vede il
transformer con `MAX_TOKEN` = N.

Accanto al classificatore due baseline senza testo: la **classe di maggioranza** del training e una **scelta a
caso** con le proporzioni del training.

Si salva **tutto**, per poter ricalcolare qualsiasi metrica senza riaddestrare:

- per ogni articolo del test: `article_id`, `domain`, `fold`, `y_true`, i **logit** `logit_l1..logit_l5`, le
  **probabilità** `p_l1..p_l5`, `y_pred`;
- `metriche.json`: per articolo, per testata col voto e per testata con la media delle probabilità —
  accuratezza, F1 macro e pesata, precisione/recall/F1 per livello, matrice di confusione, tolerant 0.5 e 1,
  errore medio in livelli, binario (soglia $l_2$/$l_3$) con TP, TN, FP, FN, precisione, recall, F1.

File: `Results/2-classificazione/<NOME>_<schema>_articoli.csv`, `<NOME>_<schema>_metriche.json`,
`<NOME>_metriche.csv` (tabella riassuntiva), `<NOME>_FATTO` quando tutto è finito.

**Sul server:** `cd ~/John/WebScience2027/2-classificazione && papermill -k mbvenv 01_classificatore_feature.ipynb ../Results/2-classificazione/esecuzioni/01_classificatore_feature.ipynb`
(di solito lo lancia `coda_weekend.py`).

In [1]:
import json
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold

from comune import CLASSI, NOME_SCHEMA, SCHEMI, SEED, carica, cartella_risultati, letture, metriche, per_testata, registratore

In [2]:
NOME = "feature"
TAGLIO_TOKEN = 0           # 0 = testo intero; N = primi N token (tokenizer di ModernBERT)
C_GRIGLIA = ""             # "" = C fisso a 1; "0.1,1,10,100" = C scelto con una validazione interna per gruppi (solo training)
SCHEMI_DA_FARE = SCHEMI
N_JOBS = 5                 # fold in parallelo
PROVA = False              # True: 40 testate, risultati in Results/2-classificazione/prova/

In [3]:
# Parameters
NOME = "feature_tuned"
C_GRIGLIA = "0.1,1,10,100"
PROVA = True


In [4]:
OUT = cartella_risultati(PROVA)
registra = registratore(OUT / f"avanzamento_{NOME}.log")
art = carica(PROVA)
if TAGLIO_TOKEN:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    enc = tok(art.text.tolist(), truncation=True, max_length=TAGLIO_TOKEN, add_special_tokens=False, return_offsets_mapping=True)
    art["text"] = [t[:o[-1][1]] if o else t for t, o in zip(art.text, enc["offset_mapping"])]
registra(f"=== {NOME}: {len(art)} articoli, {art.domain.nunique()} testate, taglio {TAGLIO_TOKEN or 'nessuno'} ===")

2026-10-18 12:26:13  === feature_tuned: 6477 articoli, 40 testate, taglio nessuno ===


## Addestramento

Per ogni fold: `fit_transform` sul training, `transform` sul test. I logit sono la `decision_function` della
regressione logistica.

In [5]:
def nuovo_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True,
                           max_features=100_000, strip_accents="unicode")


def scegli_C(tr):
    """C con la F1 macro per testata (voto) più alta in una validazione a 3 fold per gruppi, sul solo training."""
    griglia = [float(x) for x in C_GRIGLIA.split(",")]
    testate = tr.drop_duplicates("domain").reset_index(drop=True)
    punteggi = {c: [] for c in griglia}
    for itr, iva in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(testate, testate.label, testate.gruppo):
        a, b = tr[tr.domain.isin(testate.domain[itr])], tr[tr.domain.isin(testate.domain[iva])]
        v = nuovo_tfidf(); Xa, Xb = v.fit_transform(a.text), v.transform(b.text)
        for c in griglia:
            p = LogisticRegression(C=c, max_iter=1000, class_weight="balanced", random_state=SEED).fit(Xa, a.label).predict(Xb)
            t = per_testata(pd.DataFrame({"domain": b.domain.values, "y_true": b.label.values, "y_pred": p}))
            punteggi[c].append(f1_score(t.y_true, t.y_pred, average="macro", labels=range(5), zero_division=0))
    medie = {c: float(np.mean(s)) for c, s in punteggi.items()}
    return max(medie, key=medie.get), medie


def un_fold(schema, fold):
    t0 = time.time()
    tr, te = art[art[schema] != fold], art[art[schema] == fold]
    C, medie = (1.0, {}) if not C_GRIGLIA else scegli_C(tr)
    if C_GRIGLIA:
        registra(f"{schema} fold {fold}: C scelto {C} (F1 interne {', '.join(f'{k}: {v:.3f}' for k, v in medie.items())})")
    v = nuovo_tfidf()
    clf = LogisticRegression(C=C, max_iter=1000, class_weight="balanced", random_state=SEED)
    clf.fit(v.fit_transform(tr.text), tr.label)              # vocabolario, IDF e modello: solo training
    X = v.transform(te.text)                                   # test: solo transform
    registra(f"{schema} fold {fold}: fatto in {time.time() - t0:.0f} s ({len(tr)} training, {len(te)} test)")
    return te.index.to_numpy(), clf.classes_, clf.decision_function(X), clf.predict_proba(X)


def addestra(schema):
    file_out = OUT / f"{NOME}_{schema}_articoli.csv"
    if file_out.exists():
        registra(f"{schema}: già fatto, carico {file_out.name}")
        return pd.read_csv(file_out)
    logit, proba = np.full((len(art), 5), np.nan), np.zeros((len(art), 5))
    for idx, classi, z, p in Parallel(n_jobs=N_JOBS)(delayed(un_fold)(schema, f) for f in sorted(art[schema].unique())):
        logit[idx[:, None], classi[None, :]] = z
        proba[idx[:, None], classi[None, :]] = p
    pred = pd.DataFrame({"article_id": art.article_id, "domain": art.domain, "fold": art[schema], "y_true": art.label})
    for k in range(5):
        pred[f"logit_l{k + 1}"] = logit[:, k]
    for k, c in enumerate(CLASSI):
        pred[c] = proba[:, k]
    pred["y_pred"] = proba.argmax(1)
    pred.to_csv(file_out, index=False)
    json.dump(metriche(pred), open(OUT / f"{NOME}_{schema}_metriche.json", "w"), indent=1)
    return pred


pred = {s: addestra(s) for s in SCHEMI_DA_FARE}

2026-10-18 12:27:55  fold_gruppo fold 5: C scelto 100.0 (F1 interne 0.1: 0.101, 1.0: 0.082, 10.0: 0.080, 100.0: 0.113)


2026-10-18 12:28:29  fold_gruppo fold 5: fatto in 134 s (4241 training, 2236 test)


2026-10-18 12:28:35  fold_gruppo fold 2: C scelto 0.1 (F1 interne 0.1: 0.143, 1.0: 0.119, 10.0: 0.069, 100.0: 0.069)


2026-10-18 12:28:43  fold_gruppo fold 3: C scelto 0.1 (F1 interne 0.1: 0.220, 1.0: 0.112, 10.0: 0.162, 100.0: 0.187)


2026-10-18 12:28:44  fold_gruppo fold 1: C scelto 100.0 (F1 interne 0.1: 0.144, 1.0: 0.132, 10.0: 0.128, 100.0: 0.151)


2026-10-18 12:28:48  fold_gruppo fold 4: C scelto 1.0 (F1 interne 0.1: 0.148, 1.0: 0.173, 10.0: 0.141, 100.0: 0.141)


... (10 outputs left out)

2026-10-18 12:31:58  fold_testata fold 4: fatto in 168 s (5022 training, 1455 test)


2026-10-18 12:32:00  fold_testata fold 5: fatto in 171 s (4780 training, 1697 test)


2026-10-18 12:32:02  fold_testata fold 1: fatto in 173 s (5016 training, 1461 test)


2026-10-18 12:32:06  fold_testata fold 2: fatto in 176 s (5814 training, 663 test)


## Baseline senza testo

Per ogni fold, dalle testate di training: la classe più frequente, e una classe estratta a caso con le
proporzioni del training (una per testata, seed fisso).

In [6]:
def baseline(schema, modo):
    rng = np.random.default_rng(SEED)
    y = np.zeros(len(art), dtype=int)
    for fold in sorted(art[schema].unique()):
        tr, te = art[art[schema] != fold], art[art[schema] == fold]
        priori = tr.drop_duplicates("domain").label.value_counts(normalize=True).sort_index()
        if modo == "maggioranza":
            y[te.index] = priori.idxmax()
        else:
            scelte = dict(zip(te.domain.unique(), rng.choice(priori.index, te.domain.nunique(), p=priori.values)))
            y[te.index] = te.domain.map(scelte)
    return pd.DataFrame({"domain": art.domain, "y_true": art.label, "y_pred": y})


for s in SCHEMI_DA_FARE:
    for modo in ["maggioranza", "a caso"]:
        json.dump(metriche(baseline(s, modo)), open(OUT / f"baseline_{modo.replace(' ', '_')}_{s}_metriche.json", "w"), indent=1)

## Risultati

Per articolo e per testata (voto di maggioranza): accuratezza a 5 classi, F1 macro, binario e tolerant.

In [7]:
righe = []
for s in SCHEMI_DA_FARE:
    righe.append({"modello": f"TF-IDF + LR ({NOME})", "fold": NOME_SCHEMA[s], **letture(pred[s])})
    for modo in ["maggioranza", "a caso"]:
        righe.append({"modello": f"baseline: {modo}", "fold": NOME_SCHEMA[s], **letture(baseline(s, modo))})
ris = pd.DataFrame(righe).set_index(["fold", "modello"])
ris.to_csv(OUT / f"{NOME}_metriche.csv")
ris[[c for c in ris.columns if c.startswith("testata")] + ["articolo 5cl", "articolo bin"]]

testata 5cl  testata F1  testata bin  \
fold        modello                                                             
per gruppo  TF-IDF + LR (feature_tuned)        0.150       0.159        0.700   
            baseline: maggioranza              0.100       0.074        0.425   
            baseline: a caso                   0.175       0.175        0.475   
per testata TF-IDF + LR (feature_tuned)        0.250       0.247        0.700   
            baseline: maggioranza              0.075       0.046        0.375   
            baseline: a caso                   0.150       0.151        0.550   

                                         testata tol0.5  testata tol1  \
fold        modello                                                     
per gruppo  TF-IDF + LR (feature_tuned)           0.362         0.575   
            baseline: maggioranza                 0.312         0.525   
            baseline: a caso                      0.325         0.475   
per testata TF-IDF + LR (feature_tuned)           0.450         0.650   
            baseline: maggioranza                 0.225         0.375   
            baseline: a caso                      0.325         0.500   

                                         articolo 5cl  articolo bin  
fold        modello                                                  
per gruppo  TF-IDF + LR (feature_tuned)         0.161         0.624  
            baseline: maggioranza               0.079         0.412  
            baseline: a caso                    0.206         0.460  
per testata TF-IDF + LR (feature_tuned)         0.205         0.640  
            baseline: maggioranza               0.082         0.361  
            baseline: a caso                    0.146         0.593

In [8]:
t = per_testata(pred[SCHEMI_DA_FARE[0]])
LIV = ["l1", "l2", "l3", "l4", "l5"]
display(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]))
(OUT / f"{NOME}_FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

predetto,l1,l2,l3,l4,l5
vero,,,,,
l1,2,4,1,0,1
l2,3,1,1,0,3
l3,1,2,2,0,3
l4,0,1,3,1,3
l5,0,2,5,1,0


19